# Milestone 5: Competition Metric, Inference Optimization & Model Ensembling

This milestone focuses on mastering the competition metric, optimizing inference, and combining multiple models for maximum performance. You will work with fine-tuned DeBERTa and RoBERTa models, apply Softmax to obtain class probabilities, and explore ensemble techniques and Test-Time Augmentation (TTA) to improve prediction performance.

---

## Objectives

- Understand and work with the competition evaluation metric.
- Load and perform inference using fine-tuned DeBERTa and RoBERTa models.
- Apply Softmax to model logits to obtain class probabilities.
- Understand the label mapping between model outputs and answer options A-E.
- Optimize the inference process for efficient prediction.
- Combine predictions from multiple models using ensemble techniques.
- Understand and apply Test-Time Augmentation (TTA) to improve model performance.

In [1]:
import os
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForMultipleChoice, 
    TrainingArguments, 
    Trainer
)

# 1. Load Training Data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(label_map)

options = ['A', 'B', 'C', 'D', 'E']

# 2. Dataset Tokenization Function
def prepare_mcq_dataset(df, tokenizer, max_length=128):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    for idx, row in df.iterrows():
        prompt = str(row['prompt'])
        first_sentences = [prompt] * 5
        second_sentences = [str(row[opt]) for opt in options]
        
        # Tokenize sentence pairs (Prompt, Option)
        encoding = tokenizer(
            first_sentences,
            second_sentences,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        input_ids_list.append(encoding['input_ids'].numpy())
        attention_mask_list.append(encoding['attention_mask'].numpy())
        labels_list.append(row['label'])
        
    return Dataset.from_dict({
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list,
        'labels': labels_list
    })
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# ==========================================
# A. FINE-TUNE DEBERTA-V3-SMALL
# ==========================================
print("--> Training DeBERTa-v3-small on train.csv...")
deberta_name = "microsoft/deberta-v3-small"
tokenizer_deb = AutoTokenizer.from_pretrained(deberta_name)
ds_deb = prepare_mcq_dataset(train_df, tokenizer_deb)

model_deb = AutoModelForMultipleChoice.from_pretrained(deberta_name)

args_deb = TrainingArguments(
    output_dir="./deberta_checkpoints",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=1.5e-5,            # Lower learning rate prevents loss collapse
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=1.0,               # Clips exploding gradients
    fp16=False,                      # Prevents FP16 gradient underflow
    bf16=use_bf16,                   # Use BF16 if supported by hardware
    logging_steps=25,
    report_to="none"
)

trainer_deb = Trainer(
    model=model_deb,
    args=args_deb,
    train_dataset=ds_deb
)

trainer_deb.train()
trainer_deb.save_model("./deberta_finetuned")
tokenizer_deb.save_pretrained("./deberta_finetuned")
print("--> DeBERTa model fine-tuned and saved to './deberta_finetuned'")

# ==========================================
# B. FINE-TUNE ROBERTA-BASE
# ==========================================
print("\n--> Training RoBERTa-base on train.csv...")
roberta_name = "roberta-base"
tokenizer_rob = AutoTokenizer.from_pretrained(roberta_name)
ds_rob = prepare_mcq_dataset(train_df, tokenizer_rob)

model_rob = AutoModelForMultipleChoice.from_pretrained(roberta_name)

args_rob = TrainingArguments(
    output_dir="./roberta_checkpoints",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer_rob = Trainer(
    model=model_rob,
    args=args_rob,
    train_dataset=ds_rob
)

trainer_rob.train()
trainer_rob.save_model("./roberta_finetuned")
tokenizer_rob.save_pretrained("./roberta_finetuned")
print("--> RoBERTa model fine-tuned and saved to './roberta_finetuned'")

--> Training DeBERTa-v3-small on train.csv...


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension

Step,Training Loss
50,6.397837
100,4.833275
150,3.486694
200,2.628051
250,1.910398
300,1.714523
350,1.524188


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--> RoBERTa model fine-tuned and saved to './roberta_finetuned'


In [2]:
# ==============================================================================
# DLGenAI Project - Milestone 5: Inference, Ensembling & Test-Time Augmentation
# ==============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# ------------------------------------------------------------------------------
# SETUP: Load Datasets, Mappings, and Fine-Tuned Checkpoints
# ------------------------------------------------------------------------------
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
options = ['A', 'B', 'C', 'D', 'E']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Loading fine-tuned models from local checkpoints...")
tokenizer_deberta = AutoTokenizer.from_pretrained("./deberta_finetuned")
model_deberta = AutoModelForMultipleChoice.from_pretrained("./deberta_finetuned").to(device)
model_deberta.eval()
tokenizer_roberta = AutoTokenizer.from_pretrained("./roberta_finetuned")
model_roberta = AutoModelForMultipleChoice.from_pretrained("./roberta_finetuned").to(device)
model_roberta.eval()

# Core helper function for predicting option probabilities
def predict_probabilities(model, tokenizer, row, max_length=128):
    prompt = str(row['prompt'])
    first_sentences = [prompt] * 5
    second_sentences = [str(row[opt]) for opt in options]
    
    encoding = tokenizer(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    
    input_ids = encoding['input_ids'].unsqueeze(0).to(device)
    attention_mask = encoding['attention_mask'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
        
    return probs

Loading fine-tuned models from local checkpoints...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
# ==============================================================================
# QUESTION 1:
# Load the fine-tuned DeBERTa and RoBERTa models.
# For the prompt at row index 25, perform inference using each model 
# independently and apply Softmax to obtain class probabilities.
# Which answer option receives the highest probability from the DeBERTa model, 
# and what is that probability?
# ==============================================================================
row_25 = test_df.iloc[25]
probs_deberta_25 = predict_probabilities(model_deberta, tokenizer_deberta, row_25)

top1_deberta_idx = np.argmax(probs_deberta_25)
top1_deberta_option = label_map[top1_deberta_idx]
top1_deberta_prob = probs_deberta_25[top1_deberta_idx]

print(f"Q1 Answer -> {top1_deberta_option}, {top1_deberta_prob:.8f}")

Q1 Answer -> A, nan


In [4]:
# ==============================================================================
# QUESTION 2:
# Using the same sample (row index 25), average the class probabilities 
# from both models: Average Probability = [P(DeBERTa) + P(RoBERTa)] / 2
# Which answer option receives the highest averaged probability after simple 
# probability ensembling?
# ==============================================================================
probs_roberta_25 = predict_probabilities(model_roberta, tokenizer_roberta, row_25)
probs_simple_avg_25 = (probs_deberta_25 + probs_roberta_25) / 2.0
top1_simple_avg_option = label_map[np.argmax(probs_simple_avg_25)]

print(f"Q2 Answer -> {top1_simple_avg_option}")

Q2 Answer -> A


In [5]:
# ==============================================================================
# QUESTION 3:
# Apply weighted probability averaging after Softmax using the following weights:
# DeBERTa: 0.70, RoBERTa: 0.30
# Compute: P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]
# Which answer option is ranked first after weighted ensembling?
# ==============================================================================
probs_weighted_25 = (0.70 * probs_deberta_25) + (0.30 * probs_roberta_25)
top1_weighted_option = label_map[np.argmax(probs_weighted_25)]

print(f"Q3 Answer -> {top1_weighted_option}")

Q3 Answer -> A


In [6]:
# ==============================================================================
# QUESTION 4:
# Using the weighted ensemble probabilities from Q3, rank all five answer options.
# Write the final prediction exactly in Kaggle submission format.
# What is the Top-3 prediction string for row index 25? Example: C A E
# ==============================================================================
top3_indices_25 = np.argsort(probs_weighted_25)[::-1][:3]
top3_string_25 = " ".join([label_map[idx] for idx in top3_indices_25])

print(f"Q4 Answer -> {top3_string_25}")

Q4 Answer -> E D C


In [7]:
# ==============================================================================
# QUESTION 5:
# Run the weighted ensemble pipeline on every row of test.csv.
# Save the predictions in a file named submission.csv using the required Kaggle format:
# id,prediction (where prediction column contains Top-3 ranked options separated by spaces).
# Exactly how many prediction rows are present in the generated file (excluding header)?
# ==============================================================================
test_predictions = []

for idx, row in test_df.iterrows():
    p_deb = predict_probabilities(model_deberta, tokenizer_deberta, row)
    p_rob = predict_probabilities(model_roberta, tokenizer_roberta, row)
    p_weighted = (0.70 * p_deb) + (0.30 * p_rob)
    
    top3_idxs = np.argsort(p_weighted)[::-1][:3]
    top3_str = " ".join([label_map[i] for i in top3_idxs])
    test_predictions.append(top3_str)
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': test_predictions
})
submission_df.to_csv('submission.csv', index=False)

print(f"Q5 Answer -> Total Prediction Rows: {len(submission_df)}")

Q5 Answer -> Total Prediction Rows: 500


In [8]:
# ==============================================================================
# QUESTION 6:
# For the first 50 rows of test.csv, create two versions of every prompt:
# 1. Original prompt
# 2. Instruction-augmented prompt by prepending: 
#    "Answer the following multiple-choice question carefully: "
# Run inference using DeBERTa on both versions and average predicted probabilities.
# How many of the first 50 rows produce a different Top-1 prediction after applying TTA?
# ==============================================================================
def predict_probabilities_tta(model, tokenizer, row, prefix="Answer the following multiple-choice question carefully: "):
    probs_orig = predict_probabilities(model, tokenizer, row)
    
    row_aug = row.copy()
    row_aug['prompt'] = prefix + str(row['prompt'])
    probs_aug = predict_probabilities(model, tokenizer, row_aug)
    
    return (probs_orig + probs_aug) / 2.0
different_top1_count_tta = 0
for i in range(50):
    row = test_df.iloc[i]
    p_orig = predict_probabilities(model_deberta, tokenizer_deberta, row)
    p_tta = predict_probabilities_tta(model_deberta, tokenizer_deberta, row)
    
    if np.argmax(p_orig) != np.argmax(p_tta):
        different_top1_count_tta += 1

print(f"Q6 Answer -> TTA Changed Top-1 Count: {different_top1_count_tta}")

Q6 Answer -> TTA Changed Top-1 Count: 0


In [9]:
# ==============================================================================
# QUESTION 7 & 8 & 9 Cache Generation (First 100 Rows)
# ==============================================================================
deb_probs_100 = []
ens_probs_100 = []

for i in range(100):
    row = test_df.iloc[i]
    p_deb = predict_probabilities(model_deberta, tokenizer_deberta, row)
    p_rob = predict_probabilities(model_roberta, tokenizer_roberta, row)
    p_ens = (0.70 * p_deb) + (0.30 * p_rob)
    
    deb_probs_100.append(p_deb)
    ens_probs_100.append(p_ens)

In [10]:
# ==============================================================================
# QUESTION 7:
# Process the first 100 rows of test.csv and compare the Top-1 prediction from:
# 1. DeBERTa
# 2. Weighted Ensemble
# How many rows have different Top-1 predictions?
# ==============================================================================
different_top1_count_ens = 0
for i in range(100):
    if np.argmax(deb_probs_100[i]) != np.argmax(ens_probs_100[i]):
        different_top1_count_ens += 1

print(f"Q7 Answer -> Different Top-1 Predictions Count: {different_top1_count_ens}")

Q7 Answer -> Different Top-1 Predictions Count: 0


In [11]:
# ==============================================================================
# QUESTION 8:
# For the first 100 rows of test.csv, record the highest class probability (confidence) 
# predicted by:
# 1. DeBERTa
# 2. Weighted Ensemble
# For every row, compute: Confidence Gain = Ensemble Confidence - DeBERTa Confidence
# How many rows have a positive confidence gain (greater than 0)?
# ==============================================================================
positive_confidence_gain_count = 0
for i in range(100):
    conf_deb = np.max(deb_probs_100[i])
    conf_ens = np.max(ens_probs_100[i])
    if (conf_ens - conf_deb) > 0:
        positive_confidence_gain_count += 1

print(f"Q8 Answer -> Positive Confidence Gain Count: {positive_confidence_gain_count}")

Q8 Answer -> Positive Confidence Gain Count: 0


In [12]:
# ==============================================================================
# QUESTION 9:
# For the first 100 rows of test.csv, compare the Top-3 prediction strings 
# generated by:
# 1. DeBERTa alone
# 2. Weighted Ensemble
# How many rows have at least one change in their ordered Top-3 ranking after ensembling?
# Example: A C D vs. A D C
# ==============================================================================
ranking_change_count = 0
for i in range(100):
    top3_deb = [label_map[idx] for idx in np.argsort(deb_probs_100[i])[::-1][:3]]
    top3_ens = [label_map[idx] for idx in np.argsort(ens_probs_100[i])[::-1][:3]]
    
    if top3_deb != top3_ens:
        ranking_change_count += 1

print(f"Q9 Answer -> Top-3 Ranking Change Count: {ranking_change_count}")

Q9 Answer -> Top-3 Ranking Change Count: 0


In [13]:
# ==============================================================================
# QUESTION 10:
# Using the Top-3 predictions generated by your weighted ensemble for the first 
# 100 validation samples, compute the MAP@3 score.
# What is the final MAP@3 score? (Round to 4 decimal places.)
# ==============================================================================
def compute_map_at_3(pred_strings, targets):
    scores = []
    for pred, target in zip(pred_strings, targets):
        pred_list = pred.split() if isinstance(pred, str) else list(pred)
        score = 0.0
        for rank, p in enumerate(pred_list[:3]):
            if p == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

val_100 = train_df.iloc[:100]
val_preds = []

for idx, row in val_100.iterrows():
    p_deb = predict_probabilities(model_deberta, tokenizer_deberta, row)
    p_rob = predict_probabilities(model_roberta, tokenizer_roberta, row)
    p_ens = (0.70 * p_deb) + (0.30 * p_rob)
    
    top3_idxs = np.argsort(p_ens)[::-1][:3]
    top3_str = " ".join([label_map[i] for i in top3_idxs])
    val_preds.append(top3_str)

map3_score = compute_map_at_3(val_preds, val_100['answer'])
print(f"Q10 Answer -> Final MAP@3 Score: {map3_score:.4f}")

Q10 Answer -> Final MAP@3 Score: 0.3067
